# Full base-network slope calculation — data-informed candidate

**2 October 2026.** Completed new reference-profile and unit-head calculations for all 16 parent networks (5,315 links, 440 owned simple/compound domains). This is a **candidate**, separate from production hydraulic contexts and resistance/routing results.

The reference fit uses equal-cycle observation weights, the previously selected curvature strength, supported SWORD-reach P75 head-drop targets, and conservative bootstrap-derived local slope lower bounds. The latter apply **within** supported reach intervals: this is a stronger approximately-uniform-section assumption than a bound on total section drop alone. Unsupported intervals retain the old per-case profile floor explicitly. Reach sections whose current endpoint/bulk-slope ratio is outside 0.5–2 are excluded to reduce break-crossing problems; this is a screening choice, not an automatic geomorphic breakpoint detector.

Units use the maximum graph-feasible common fraction of local target slopes, then minimize normalized slope deviations while holding boundary heads fixed. Critical-path heads are eliminated analytically before optimization. Numerical head-bound violations are at most about 13 micrometres; final slopes are all positive and shared node heads agree exactly.

**Important finding:** some old smoothing strengths produce much smoother profiles when the objective is solved more fully. Controls below separate this from the new constraints. Fewer low slopes is not sufficient reason to adopt a candidate. The new smoothing/constraint combination still needs selection using withheld cycles. Case 46's old post-fit manual bridge is not reapplied in this candidate; inspect 370–410 km explicitly.

[Method and results report](../assets/data_informed_base_slopes_20261002/README.md).

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'RAPID/src').is_dir())
OUT=ROOT/'RAPID/assets/data_informed_base_slopes_20261002'
summary=pd.read_csv(OUT/'summary.csv',dtype={'example':str});summary.example=summary.example.str.zfill(2)
assert len(summary)==16 and (summary.status=='complete').all()
assert summary.n_links.sum()==5315 and summary.n_units.sum()==440
pio.renderers.default='plotly_mimetype'
display(summary[['example','n_links','n_units','n_section_targets','profile_supported_length_pct','old_low_links','new_low_links','new_min_slope','head_loss_change_pct','old_observation_rms_m','new_observation_rms_m']])
print('Low slopes (<1e-5):',summary.old_low_links.sum(),'→',summary.new_low_links.sum())
display(pd.read_csv(OUT/'validation.csv'))
display(pd.read_csv(OUT/'unit_bound_validation.csv'))

## Controls: new constraints versus a more fully solved smoothing objective

These controls omit the new section targets and data-derived floors, retaining the old global profile floor and smoothing strength. One uses equal observation weights; the other uses equal cycle weights. They do not reapply case 46's old manual post-fit bridge. All RMS values are in-sample WSE residuals after acquisition offsets—not independent validation.

A control that differs from the saved profile shows that changing the solver/degree of convergence matters independently of the new section constraints. The objective table includes the curvature penalty: a smaller penalized objective can coincide with a worse raw observation RMS. Old smoothing strengths must therefore be reconsidered under the new converged fitting procedure.

In [ ]:
controls=pd.read_csv(OUT/'profile_controls.csv',dtype={'example':str});controls.example=controls.example.str.zfill(2)
comparison=summary.set_index('example')[['old_observation_rms_m','new_observation_rms_m']].join(controls.pivot(index='example',columns='mode',values='observation_rms_m'))
display(comparison)
display(pd.read_csv(OUT/'objective_controls.csv').pivot(index='example',columns='variant',values='total_objective'))

In [ ]:
def inspect(ex):
    path=OUT/ex
    f=pd.read_parquet(path/'reference_profile.parquet')
    o=pd.read_parquet(path/'reference_profile_observations.parquet')
    links=pd.read_parquet(path/'candidate_link_hydraulic_context.parquet')
    nodes=pd.read_parquet(path/'candidate_node_hydraulic_context.parquet').set_index('id_node')
    controls=pd.read_parquet(path/'control_equal_cycle_profile.parquet')
    fig=go.Figure()
    for aid,g in o.groupby('acquisition_id',sort=False):
        g=g.sort_values('reference_section_dist_out_m');r=g.iloc[0]
        fig.add_scatter(x=g.reference_section_dist_out_m/1000,y=g.wse_m,mode='lines+markers',name=f"Cycle {int(r.cycle):03d}, pass {int(r['pass']):03d}",visible='legendonly',opacity=.4,line=dict(width=1),marker=dict(size=2))
    env=o.groupby('reference_section_dist_out_m').wse_m.quantile([.1,.9]).unstack().sort_index()
    fig.add_scatter(x=env.index/1000,y=env[.9],line=dict(width=0),showlegend=False,hoverinfo='skip')
    fig.add_scatter(x=env.index/1000,y=env[.1],line=dict(width=0),fill='tonexty',fillcolor='rgba(120,120,120,.16)',name='Raw screened P10–P90 envelope')
    fig.add_scatter(x=f.reference_section_dist_out_m/1000,y=f.old_fitted_wse_m,name='Previous promoted profile',line=dict(color='#555555',dash='dash'))
    fig.add_scatter(x=f.reference_section_dist_out_m/1000,y=f.fitted_wse_m,name='New candidate profile',line=dict(color='#d62728',width=3))
    fig.add_scatter(x=controls.reference_section_dist_out_m/1000,y=controls.fitted_wse_m,name='Control: old floor, equal-cycle weights',visible='legendonly',line=dict(color='#2563eb'))
    fig.add_scatter(x=f.reference_section_dist_out_m/1000,y=f.static_wse_m,name='Static reference',visible='legendonly',line=dict(color='#9467bd',dash='dot'))
    fig.update_layout(title=f'Case {ex} — full reference-profile calculation',xaxis_title='Distance to outlet (km)',yaxis_title='WSE (m)',height=550,template='plotly_white')
    fig.show()
    middle=(links.id_us_node.map(nodes.reference_section_dist_out_m)+links.id_ds_node.map(nodes.reference_section_dist_out_m))/2000
    fig=go.Figure()
    for column,label,color in [('previous_slope_used','Previous link slope','#777777'),('slope_used','New link slope','#d62728')]:
        fig.add_scatter(x=middle,y=links[column],mode='markers',name=label,marker=dict(size=5,color=color,opacity=.65),customdata=np.c_[links.id_link,links.ownership_kind,links.actual_length_m],hovertemplate='Link %{customdata[0]} · %{customdata[1]}<br>Length %{customdata[2]:.1f} m<br>Slope %{y:.4g}<extra></extra>')
    fig.add_scatter(x=middle,y=links.unit_target_slope,mode='markers',name='Unit target before graph relaxation',visible='legendonly',marker=dict(size=4,color='#2563eb'))
    fig.add_scatter(x=middle,y=links.unit_lower_envelope_slope,mode='markers',name='Graph-feasible lower envelope',visible='legendonly',marker=dict(size=4,color='#16a34a'))
    fig.add_hline(y=1e-5,line_dash='dot',line_color='#999999')
    fig.update_layout(title=f'Case {ex} — physical link slopes after unit solves',xaxis_title='Projected link midpoint (km)',yaxis_title='Slope (m/m)',yaxis_type='log',height=430,template='plotly_white')
    fig.show()
    display(summary[summary.example==ex].drop(columns=['seconds']))
    display(pd.read_csv(path/'unit_solver_audit.csv')[['kind','domain','n_links','boundary_drop_m','max_feasible_target_fraction','n_critical_fixed_nodes','iterations']])
    low=links[links.slope_used<1e-5]
    if len(low):
        display(Markdown('**Remaining low-slope links**'))
        display(low[['id_link','ownership_kind','ownership_domain_id','actual_length_m','previous_slope_used','slope_used','unit_target_slope','unit_lower_envelope_slope']])

## Case 02

In [ ]:
inspect("02")

## Case 03

In [ ]:
inspect("03")

## Case 05

In [ ]:
inspect("05")

## Case 11

In [ ]:
inspect("11")

## Case 13

In [ ]:
inspect("13")

## Case 15

In [ ]:
inspect("15")

## Case 21

In [ ]:
inspect("21")

## Case 22

In [ ]:
inspect("22")

## Case 37

In [ ]:
inspect("37")

## Case 41

In [ ]:
inspect("41")

## Case 45

In [ ]:
inspect("45")

## Case 46

In [ ]:
inspect("46")

## Case 48

In [ ]:
inspect("48")

## Case 54

In [ ]:
inspect("54")

## Case 55

In [ ]:
inspect("55")

## Case 56

In [ ]:
inspect("56")

## Interpretation before any promotion

All numerical calculations completed. Links below 1e-5 fell from 57 to 26: 7 in case 11, 1 in case 15, 18 in case 46. The candidate therefore does not eliminate every low slope, nor should a data-based method guarantee that.

Inspect cases 02, 22, 48, 54 and 56 for increased observation residuals; controls show much of the degradation in several cases occurs even without new section constraints. Case 21 has a substantial head-loss increase. Next select smoothing strength under the new solver and validate the section assumptions using withheld cycles. This notebook records a completed candidate calculation, not accepted hydraulic inputs or evidence of routing stability.